In [ ]:
# ============================================================
#  CLASE: WORKING WITH LLMs - API DE OPENAI
#  Rol: Analista de Datos e IA
#  Versión didáctica para alumnos principiantes
# ============================================================

from openai import OpenAI
import getpass
import json

# ------------------------------------------------------------
#  CONFIGURACIÓN
# ------------------------------------------------------------
api_key = getpass.getpass("🔑 Ingresa tu API KEY de OpenAI: ")
client = OpenAI(api_key=api_key)
print(" Listo!\n")

# Definimos UN solo rol que se reutiliza en todos los ejemplos
ROL = "Eres un analista de datos amable que explica resultados de forma clara y simple."

# Datos de ejemplo: ventas mensuales de una pequeña tienda
DATOS = """
Ventas del último trimestre:
- Enero:   $12.000
- Febrero: $9.500
- Marzo:   $15.200
"""


In [ ]:

# ============================================================
#  BLOQUE 1: PRIMERA LLAMADA A LA API
# ------------------------------------------------------------
#  Conceptos: model, system prompt, user prompt, temperature
# ============================================================
#print("=" * 50)
#print("  BLOQUE 1: Tu primera llamada al LLM")
#print("=" * 50)

respuesta = client.chat.completions.create(
    model="gpt-4o-mini",                 # Modelo: rápido y económico
    messages=[
        {"role": "system", "content": ROL},        # Define cómo se comporta
        {"role": "user", "content": f"Analiza estas ventas:\n{DATOS}"}
    ],
    temperature=0.3,                     # 0 = preciso | 1 = creativo
    max_tokens=800,                      # Largo máximo de la respuesta
)

#print(respuesta.choices[0].message.content)
#print(f"\n📊 Tokens usados: {respuesta.usage.total_tokens}")




In [ ]:
# ============================================================
#  BLOQUE 2: EFECTO DE TEMPERATURE
# ------------------------------------------------------------
#  Mismo prompt, distinta temperature → distinto estilo
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 2: Cómo cambia la respuesta con temperature")
print("=" * 50)

pregunta = "Sugiere un nombre creativo para un dashboard de ventas."

for temp in [0, 1.5]:
    print(f"\n--- temperature = {temp} ---")
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ROL},
            {"role": "user", "content": pregunta}
        ],
        temperature=temp,
        max_tokens=800,
    )
    print(r.choices[0].message.content)

print("\n💡 temperature baja = consistente | temperature alta = creativo")



In [ ]:

# ============================================================
#  BLOQUE 3: SALIDA EN JSON (para usar en un sistema)
# ------------------------------------------------------------
#  Cuando queremos integrar el LLM con código o una BD
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 3: Respuesta en formato JSON")
print("=" * 50)

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": ROL + " Devuelve SIEMPRE un JSON con: tendencia, mes_mayor, mes_menor, recomendacion."
        },
        {"role": "user", "content": DATOS}
    ],
    temperature=0,
    response_format={"type": "json_object"},
    max_tokens=200,
)

datos = json.loads(r.choices[0].message.content)
print(json.dumps(datos, indent=2, ensure_ascii=False))
print("\n💡 Este JSON ya se puede guardar en una base de datos o mostrar en un dashboard.")





In [ ]:

# ============================================================
#  BLOQUE 4: CONVERSACIÓN CON MEMORIA (multi-turno)
# ------------------------------------------------------------
#  El LLM no recuerda nada → tenemos que enviarle el historial
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 4: Conversación con contexto")
print("=" * 50)
print("Escribe tus preguntas. Escribe 'salir' para terminar.\n")

historial = [
    {"role": "system", "content": ROL},
    {"role": "user", "content": f"Te paso estos datos:\n{DATOS}"}
]

# Primera respuesta del bot para arrancar
r = client.chat.completions.create(
    model="gpt-4o-mini", messages=historial, temperature=0.3, max_tokens=150
)
historial.append({"role": "assistant", "content": r.choices[0].message.content})
print(f"🤖 Analista: {r.choices[0].message.content}\n")

# Loop de conversación
while True:
    pregunta = input("👤 Tú: ").strip()
    if pregunta.lower() == "salir":
        break
    historial.append({"role": "user", "content": pregunta})

    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=historial, temperature=0.3, max_tokens=200
    )
    respuesta = r.choices[0].message.content
    historial.append({"role": "assistant", "content": respuesta})
    print(f"🤖 Analista: {respuesta}\n")

print(f"\n Cada turno reenvía TODO el historial. Por eso decimos que el LLM es 'sin memoria'.")
print("\n ¡Fin de la clase! Ya viste los 4 conceptos clave para trabajar con LLMs.")